IMPORTS

In [1]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
import numpy as np

STOPWORD OCEAN

In [2]:
stop = set(stopwords.words("english"))

LOADING DATA

In [3]:
df = pd.read_excel(r'Bank.xlsx')

CATEGORIZATION

In [16]:
df_fin = df[df['Sense of the word "Bank"'] == "Financial Institution"]
df_rv = df[df['Sense of the word "Bank"'] == "River Border"]
df_test = df[df['Sense of the word "Bank"'] == "?"]

REMOVING STOPWORDS

In [6]:
def clean(text):
    return [w for w in word_tokenize(text.lower()) if w.isalpha() and w not in stop]

COUNTS OF SENSES

In [17]:
fin_count = Counter([w for sentence in df_fin["Sentences for Training "] for w in clean(sentence)])
rv_count = Counter([w for sentence in df_rv["Sentences for Training "] for w in clean(sentence)])

PRIORS

In [18]:
p_fin = len(df_fin) / len(df)
r_fin = len(df_rv) / len(df)

In [23]:
print(len(df_rv))

48


VOCABULARY

In [24]:
V = len(set(fin_count.keys())) + len(set(rv_count.keys()))

In [29]:
def classify(sentence):
    words = clean(sentence)
    log_fin = np.log(p_fin)
    log_rv = np.log(r_fin)

    for w in words:
        log_fin += np.log(
                        (
                            fin_count.get(w,0)+1
                        ) / (
                            sum(fin_count.values()) + (V)
                        )
                        )
        
        log_rv += np.log(
                        (
                            rv_count.get(w,0)+1
                        ) / (
                            sum(rv_count.values()) + (V)
                        )
                        )
        
    return "Financial Institution" if log_fin > log_rv else "River Border"

In [30]:

for sentence in df_test["Sentences for Training "]:
    print(sentence, "→", classify(sentence))

The children built a dam on the bank of the river using rocks and sticks. → River Border
We need to withdraw some cash from the bank for groceries. → Financial Institution
I need to update my contact information with the bank.  → Financial Institution
The bank provides online banking services for convenience.  → Financial Institution
The beavers constructed a dam along the bank of the river. → River Border
I need to check my transaction history at the bank.  → Financial Institution
She works as a financial consultant at the bank.  → Financial Institution
